In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
# Import libraries

from src.dependence.dependence import cov_matrix
from src.data.returns import split_returns
from src.dependence.copula_modelling import load_vine_model
from src.data.wrangler import get_kept_tickers
from src.optimisation.hmv.clustering import clustering_matrix, dependence_matrix
from src.optimisation.hmv.seriation import quasi_diagonalisation
from src.optimisation.hmv.recursive_bisection import heuristic_optimisation, gamma_sensitivity, plot_gamma_sensitivity
 
from src.data.returns import * 
from src.optimisation.mvp.mvp_solver import *
from src.optimisation.hrp.hrp_weights import hrp_weights
from src.optimisation.eqw.eq_weights import equal_weights

from src.performance.rolling_weights import (
    rolling_weights,
    plot_rolling_weights,
    plot_turnover_comparison
)

from src.performance.portfolio_metrics import portfolio_metrics, sub_period_metrics

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [13]:
t_matrix = dependence_matrix(1)
t_matrix.head()

Loading local data...


,ABG,ACT,ADH,AFT,AGL,AME,ANG,APN,ART,BEL,...,SLM,SNT,SPG,SPP,TFG,TKG,TRU,VOD,WBO,WHL
ABG,1.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0932,0.0
ACT,0.0,1.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0467,0.0,0.0000,0.0
ADH,0.0,0.0,1.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0000,0.0
AFT,0.0,0.0,0.0,1.0,0.0,0.0,0.0000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0000,0.0
AGL,0.0,0.0,0.0,0.0,1.0,0.0,0.5776,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0000,0.0


In [10]:
t_matrix.to_csv('../data/output/results/matrix_t1.csv')


OSError: Cannot save file into a non-existent directory: '..\data\output\results'

In [15]:
load_vine_model()

<pyvinecopulib.Vinecop> Vinecop model with 55 variables
tree edge conditioned variables                                                                                                                                                                                     conditioning variables var_types   family rotation   parameters  df   tau 
   1    1                29, 37                                                                                                                                                                                                                 c, c  Student        0   0.41, 7.19 2.0  0.27 
   1    2                 8, 41                                                                                                                                                                                                                 c, c  Student        0   0.38, 6.41 2.0  0.25 
   1    3                27, 32                                                    

In [16]:
train, test = split_returns()

Loading local data...


In [17]:
hmv_weights_train = heuristic_optimisation(train)
hmv_returns_test = test @ hmv_weights_train



Loading local data...
Loading local data...
Loading local data...


In [18]:
hrp_weights_train = hrp_weights(train)
hrp_returns_test = test @ hrp_weights_train


In [19]:
eqw_weights_train = equal_weights(returns=train)
eqw_returns_test = test @ eqw_weights_train

In [20]:
mvp_weights_train = mvp_weights(train.cov())
mvp_returns_test = test @ mvp_weights_train

In [21]:
# Collect all return series into one DataFrame
returns_df = pd.DataFrame({
    'HMV-Copula' : hmv_returns_test,
    'HRP'        : hrp_returns_test,
    'Equal Weight': eqw_returns_test,
    'MVP'        : mvp_returns_test
})

# Confirm shape and check for nulls
print(returns_df.shape)
print(returns_df.isnull().sum())
print(returns_df.describe())

(974, 4)
HMV-Copula      0
HRP             0
Equal Weight    0
MVP             0
dtype: int64
       HMV-Copula         HRP  Equal Weight         MVP
count  974.000000  974.000000    974.000000  974.000000
mean     0.000832    0.000690      0.000537    0.000676
std      0.008149    0.007466      0.015396    0.006961
min     -0.039932   -0.041600     -0.245559   -0.035090
25%     -0.003286   -0.002916     -0.003669   -0.003191
50%      0.000788    0.000547      0.000758    0.000655
75%      0.005231    0.004671      0.005230    0.004447
max      0.044374    0.061932      0.091443    0.075830


In [22]:
# Apply to every column uniformly
table1 = pd.DataFrame(
    {col: portfolio_metrics(returns_df[col]) for col in returns_df.columns}
).T

table1.index.name = 'Strategy'
print(table1.to_string())

              Annualised Return (%)  Annualised Vol (%)  Sharpe Ratio  Max Drawdown (%)  Calmar Ratio  Sortino Ratio
Strategy                                                                                                            
HMV-Copula                    20.96               12.94         1.060            -10.71         1.957          1.098
HRP                           17.39               11.85         0.855             -8.88         1.958          0.891
Equal Weight                  13.53               24.44         0.257            -27.81         0.487          0.219
MVP                           17.03               11.05         0.885            -10.62         1.603          0.949


In [23]:
# LaTeX (paste directly into your thesis)
print(table1.to_latex(float_format="%.2f", bold_rows=True))

# Excel (easier to format manually)
table1.to_excel('table1_performance.xlsx')

\begin{tabular}{lrrrrrr}
\toprule
 & Annualised Return (%) & Annualised Vol (%) & Sharpe Ratio & Max Drawdown (%) & Calmar Ratio & Sortino Ratio \\
Strategy &  &  &  &  &  &  \\
\midrule
\textbf{HMV-Copula} & 20.96 & 12.94 & 1.06 & -10.71 & 1.96 & 1.10 \\
\textbf{HRP} & 17.39 & 11.85 & 0.85 & -8.88 & 1.96 & 0.89 \\
\textbf{Equal Weight} & 13.53 & 24.44 & 0.26 & -27.81 & 0.49 & 0.22 \\
\textbf{MVP} & 17.03 & 11.05 & 0.89 & -10.62 & 1.60 & 0.95 \\
\bottomrule
\end{tabular}



In [26]:
table2 = sub_period_metrices(returns_df=returns_df)
print(table2.to_string())

NameError: name 'sub_period_metrices' is not defined

In [ ]:
results = gamma_sensitivity(
    train_returns=train,
    test_returns=test,
    gamma_grid=np.linspace(0, 1, 51)
)

In [ ]:
optimal_gamma, optimal_sharpe = plot_gamma_sensitivity(results)

In [ ]:
final_weights = heuristic_optimisation(
    train_returns=train,
    gamma=optimal_gamma
)

In [ ]:
# ── Cell: Figure 4 — Rolling Weight Stability ─────────────────────────────
from src.performance.rolling_weights import (
    rolling_weights,
    plot_rolling_weights,
    plot_turnover_comparison
)
from src.data.returns import get_log_returns

# use full returns for the rolling window — not just the test split
full_returns = get_log_returns()

# compute rolling weights (this takes a few minutes)
weight_dfs = rolling_weights(
    returns_df   = full_returns,
    train_window = 504,          # 2-year training window
    refit_freq   = 63,           # refit every quarter
    gamma        = 0.0           # replace with optimal_gamma from Figure 3
)




In [ ]:
# Figure 4a — stacked area charts
plot_rolling_weights(weight_dfs, top_n=10)

# Figure 4b — turnover bar chart
# plot_turnover_comparison(weight_dfs)

In [ ]:
plot_turnover_comparison(weight_dfs)

In [ ]:
# ── Cell: Figure 6 — Cumulative Returns ───────────────────────────────────
from src.performance.cumulative_returns import (
    plot_cumulative_returns,
    print_crisis_performance
)

# plot Figure 6 — uses returns_df already built earlier in the notebook
plot_cumulative_returns(returns_df, log_scale=True)